# NOTEBOOK DASHBOARD

## DF_EDA

In [1]:
import pandas as pd

df_eda = pd.read_csv(r"..\00_Data\00_Processed\df_eda.csv")

In [ ]:
#create the "other" field for the least represented races

df_eda['race'] = df_eda['race'].replace({'asian': 'other', 'native american': 'other'})

In [3]:
df_eda.to_csv(r'..\00_Data\00_Processed\df_dashboard.csv', index=False)

## DF_EVAL

In [4]:
df_eval = pd.read_csv(r"..\00_Data\00_Processed\df_eval.csv")

In [ ]:
#rename the "sex" variable

df_eval['sex'] = df_eval['sex'].replace({0: 'male', 1: 'female'})

In [7]:
df_sin_outliers = pd.read_csv(r"..\00_Data\00_Processed\df_sin_outliers.csv")

In [ ]:
#perform a merge on person_id

df_eval = df_eval.merge(
    df_sin_outliers[['person_id','decile_score']],
    on='person_id',
    how='left'
)

#generate new variables to compare models

df_eval['score_propublica'] = df_eval['decile_score'] / 10

df_eval['y_pred_propublica'] = (df_eval['score_propublica'] >= 0.5).astype(int)

In [ ]:
#convert the model scores into deciles
df_eval['decil_modelo'] = pd.qcut(
    df_eval['y_score'].rank(method='first'),
    10,
    labels=False
)


df_eval['decil_propublica'] = pd.qcut(
    df_eval['score_propublica'].rank(method='first'),
    10,
    labels=False
)

df_eval['decil_modelo'] = df_eval['decil_modelo'] + 1
df_eval['decil_propublica'] = df_eval['decil_propublica'] + 1

In [ ]:
#create the dataframe with the necessary information

df_eval_long = pd.concat([

    
    df_eval[['person_id','race','sex','age_cat','y_true','decil_modelo','y_score']]
        .rename(columns={
            'decil_modelo':'decil',
            'y_score':'score'
        })
        .assign(modelo='propuesta_modelo'),

    
    df_eval[['person_id','race','sex','age_cat','y_true','decil_propublica','score_propublica']]
        .rename(columns={
            'decil_propublica':'decil',
            'score_propublica':'score'
        })
        .assign(modelo='propublica')

])

In [ ]:
#create score intervals

df_eval_long['score_bin'] = pd.cut(
    df_eval_long['score'],
    bins=[0,0.2,0.4,0.6,0.8,1]
)

In [ ]:
df_eval_long.to_csv(r'..\00_Data\00_Processed\df_eval_long.csv', index=False)

## DF_FAIRNESS

In [ ]:
#calculate false positive and true positive rates to assess fairness

df_fairness_modelo = df_eval.groupby(['race','sex','age_cat']).apply(
    lambda g: pd.Series({
        'FPR': ((g['y_pred']==1)&(g['y_true']==0)).sum() / max((g['y_true']==0).sum(),1),
        'TPR': ((g['y_pred']==1)&(g['y_true']==1)).sum() / max((g['y_true']==1).sum(),1)
    })
).reset_index()

df_fairness_modelo['modelo'] = 'propuesta_modelo'

df_fairness_propublica = df_eval.groupby(['race','sex','age_cat']).apply(
    lambda g: pd.Series({
        'FPR': ((g['y_pred_propublica']==1)&(g['y_true']==0)).sum() / max((g['y_true']==0).sum(),1),
        'TPR': ((g['y_pred_propublica']==1)&(g['y_true']==1)).sum() / max((g['y_true']==1).sum(),1)
    })
).reset_index()

df_fairness_propublica['modelo'] = 'propublica'

df_fairness = pd.concat([
    df_fairness_modelo,
    df_fairness_propublica
])

C:\Users\JAIME\AppData\Local\Temp\ipykernel_138020\1828374102.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_fairness_modelo = df_eval.groupby(['race','sex','age_cat']).apply(
C:\Users\JAIME\AppData\Local\Temp\ipykernel_138020\1828374102.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_fairness_propublica = df_eval.groupby(['race','sex','age_cat']).apply(


In [ ]:
#generate possible combinations based on demographic variables

df_fairness['grupo'] = (
    df_fairness['race'] + ' | ' +
    df_fairness['sex'] + ' | ' +
    df_fairness['age_cat']
)

In [ ]:
df_fairness.to_csv(r'..\00_Data\00_Processed\df_fairness.csv', index=False)